# SC 612 104 Essential Data Science
## SQL ขั้นสูง: JOIN, Subquery, เชื่อม Python กับ SQLite, และ Mini-Exercise

**ผู้สอน:** อ.ดร.พิชญา วิรัชโชติเสถียร (Pitchaya Wiratchotisatian)
**ภาควิชาสถิติ คณะวิทยาศาสตร์ มหาวิทยาลัยขอนแก่น**

---

### เนื้อหาใน Notebook นี้

**ส่วนที่ 1: JOIN**
1. ทำไมต้องมี JOIN
2. `INNER JOIN` — เมื่อไหร่ใช้
3. `LEFT JOIN` — เมื่อไหร่ใช้
4. `RIGHT JOIN` — เมื่อไหร่ใช้ (และทำไม SQLite ไม่มี RIGHT JOIN ตรงๆ)
5. Subquery เบื้องต้น

**ส่วนที่ 2: เชื่อม Python กับ SQLite**\
6. `sqlite3` พื้นฐาน\
7. `pandas.read_sql_query()` — ได้ DataFrame ตรงจาก SQL\
8. Parameterized Query — ป้องกัน SQL Injection

**ส่วนที่ 3: End-to-end Mini-Exercise**\
9. DB → SQL → DataFrame → Clean → Visualize (ครบวงจรในไฟล์เดียว)

**แบบฝึกหัดท้ายเรื่อง** 

> 📌 **ก่อนเริ่ม:** Notebook นี้ใช้ไฟล์ `online_store.db` (อัปโหลดเข้า Colab ถ้าจำเป็น)


In [ ]:
# ถ้าใช้ Google Colab และยังไม่ได้อัปโหลดไฟล์ ให้รันเซลล์นี้เพื่ออัปโหลด online_store.db
try:
    from google.colab import files
    uploaded = files.upload()   # เลือก online_store.db (สร้างไว้จาก Notebook ที่ 1)
except ImportError:
    print("ไม่ได้รันบน Colab - ข้ามขั้นตอนนี้ (ตรวจสอบว่า online_store.db อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)")

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ตั้งค่าฟอนต์ให้รองรับภาษาไทยในกราฟ
import matplotlib.font_manager as fm
thai_fonts = ["Loma", "TH Sarabun New", "Tahoma", "Noto Sans Thai"]
available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in thai_fonts:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

conn = sqlite3.connect("online_store.db")
cursor = conn.cursor()

tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("เชื่อมต่อฐานข้อมูลสำเร็จ ตารางที่มี:", [t[0] for t in tables])

---
# ส่วนที่ 1: JOIN

## 1. ทำไมต้องมี JOIN?

จากไฟล์ที่แล้ว ฐานข้อมูลถูกออกแบบให้แยกข้อมูลเป็น 4 ตาราง (Normalization) เพื่อลดความซ้ำซ้อน — แต่เวลาต้องการ**ดูข้อมูลรวมกัน** (เช่น "ชื่อลูกค้าคนไหนสั่งซื้ออะไรบ้าง") ต้อง "เชื่อม" ตารางเข้าด้วยกันก่อน — นี่คือหน้าที่ของ **JOIN**

เทียบกับที่เรียนมา: `JOIN` ใน SQL คือสิ่งเดียวกับ `pd.merge()` ของ pandas ที่เรียนมาจาก Session 9 — ทั้งสองทำสิ่งเดียวกัน แค่ใช้ใน 2 บริบทต่างกัน (SQL ทำงานกับฐานข้อมูล, pandas ทำงานกับ DataFrame ที่โหลดเข้าหน่วยความจำแล้ว)

## 2. `INNER JOIN` — เชื่อมเฉพาะแถวที่ตรงกันทั้งสองตาราง

### โครงสร้าง

```sql
SELECT คอลัมน์
FROM ตารางA
INNER JOIN ตารางB ON ตารางA.คีย์ = ตารางB.คีย์
```

`INNER JOIN` คืนผลลัพธ์เฉพาะแถวที่มีค่าคีย์**ตรงกันทั้งสองตาราง** — ถ้าฝั่งใดฝั่งหนึ่งไม่มีคู่ตรงกัน แถวนั้นจะไม่ปรากฏในผลลัพธ์เลย (เทียบกับ `how="inner"` ของ `pd.merge()` ที่เรียนมา — เป็นค่า default ทั้งของ SQL JOIN และ `pd.merge()`)


In [ ]:
result = pd.read_sql_query("""
SELECT c.first_name, c.last_name, o.order_id, o.order_date, o.status
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.order_date
LIMIT 10
""", conn)
result

**สังเกต:** เราใช้ `c` และ `o` เป็น **alias** (ชื่อย่อ) ของตาราง `customers` และ `orders` — ช่วยให้เขียน query สั้นลงเมื่อต้องอ้างถึงคอลัมน์จากหลายตาราง โดยเฉพาะเมื่อทั้งสองตารางมีคอลัมน์ชื่อเดียวกัน (เช่น `customer_id` ที่มีอยู่ในทั้ง `customers` และ `orders`)

### 2.1 จุดสำคัญของ `INNER JOIN`: ลูกค้าที่ไม่มีคำสั่งซื้อจะไม่ปรากฏเลย


In [ ]:
# เช็คจำนวนลูกค้าทั้งหมด เทียบกับจำนวนลูกค้าที่ปรากฏใน INNER JOIN กับ orders
total_customers = pd.read_sql_query("SELECT COUNT(*) AS n FROM customers", conn)["n"][0]

customers_with_orders = pd.read_sql_query("""
SELECT COUNT(DISTINCT c.customer_id) AS n
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
""", conn)["n"][0]

print(f"จำนวนลูกค้าทั้งหมด: {total_customers}")
print(f"จำนวนลูกค้าที่ปรากฏใน INNER JOIN กับ orders: {customers_with_orders}")
print(f"ลูกค้าที่ \"หายไป\" จาก INNER JOIN (เพราะไม่มีคำสั่งซื้อเลย): {total_customers - customers_with_orders} คน")

**🔍 นี่คือข้อจำกัดสำคัญของ `INNER JOIN`:** ถ้าต้องการรายงานที่ต้อง**เห็นลูกค้าทุกคน** (รวมคนที่ยังไม่เคยสั่งซื้อ) `INNER JOIN` จะไม่เหมาะ ต้องใช้ `LEFT JOIN` แทน (หัวข้อถัดไป)

## 3. `LEFT JOIN` — เก็บทุกแถวจากตารางซ้าย แม้ไม่มีคู่ตรงกัน

### โครงสร้าง

```sql
SELECT คอลัมน์
FROM ตารางA          -- ตารางซ้าย (เก็บทุกแถว)
LEFT JOIN ตารางB ON ตารางA.คีย์ = ตารางB.คีย์
```

ถ้าแถวใดของตารางซ้ายไม่มีคู่ตรงกันในตารางขวา ผลลัพธ์คอลัมน์จากตารางขวาจะเป็น `NULL` (เทียบกับ `NaN` ของ pandas ที่เรียนมา) แต่แถวนั้น**ไม่ถูกตัดทิ้ง**


In [ ]:
result = pd.read_sql_query("""
SELECT c.customer_id, c.first_name, c.last_name, o.order_id, o.order_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id
""", conn)

print("จำนวนแถวทั้งหมด:", len(result))
print("\nลูกค้าที่ไม่มีคำสั่งซื้อเลย (order_id เป็น NULL/NaN):")
result[result["order_id"].isnull()]

**🔍 สังเกต:** ตอนนี้เห็นลูกค้าที่ไม่มีคำสั่งซื้อเลยแล้ว (คอลัมน์ `order_id`, `order_date` เป็น `None`/`NaN`) — นี่คือสิ่งที่ `INNER JOIN` ไม่แสดงให้เห็น

### 3.1 ตัวอย่างการใช้งานจริง: หาลูกค้าที่ "เงียบ" (ไม่เคยสั่งซื้อเลย) เพื่อทำการตลาด


In [ ]:
inactive_customers = pd.read_sql_query("""
SELECT c.customer_id, c.first_name, c.last_name, c.email, c.signup_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
""", conn)

print(f"พบลูกค้าที่ยังไม่เคยสั่งซื้อเลย {len(inactive_customers)} คน:")
inactive_customers

**เทคนิคสำคัญ:** การหา "แถวที่ไม่มีคู่ตรงกัน" ทำได้โดย `LEFT JOIN` แล้วกรองด้วย `WHERE คอลัมน์จากตารางขวา IS NULL` — เป็นเทคนิคที่ใช้บ่อยมากในการหาข้อมูลที่ "ขาด" ความสัมพันธ์ (เช่น ลูกค้าที่ไม่ได้ใช้งาน, สินค้าที่ไม่เคยขาย)

## 4. `RIGHT JOIN` — เก็บทุกแถวจากตารางขวา

### โครงสร้าง

```sql
SELECT คอลัมน์
FROM ตารางA
RIGHT JOIN ตารางB ON ตารางA.คีย์ = ตารางB.คีย์   -- เก็บทุกแถวจาก ตารางB (ขวา)
```

`RIGHT JOIN` ทำงานตรงข้ามกับ `LEFT JOIN` — เก็บทุกแถวจากตารางขวาแทน

**⚠️ ข้อสำคัญ: SQLite เวอร์ชันเก่าไม่รองรับ `RIGHT JOIN` โดยตรง** (รองรับเฉพาะ SQLite 3.39+) — วิธีแก้ทั่วไปคือ**สลับลำดับตาราง**แล้วใช้ `LEFT JOIN` แทน ซึ่งให้ผลลัพธ์เหมือนกันทุกประการ


In [ ]:
# RIGHT JOIN แบบเทียบเท่า: หาสินค้าทั้งหมด (รวมที่ไม่เคยถูกสั่งซื้อ) โดยสลับตารางแล้วใช้ LEFT JOIN
# เทียบเท่ากับ: SELECT ... FROM order_items RIGHT JOIN products ON ... (ถ้า SQLite รองรับ RIGHT JOIN)
result = pd.read_sql_query("""
SELECT p.product_id, p.product_name, oi.order_item_id, oi.quantity
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
ORDER BY p.product_id
""", conn)

print("สินค้าที่ไม่เคยถูกสั่งซื้อเลย (order_item_id เป็น NULL):")
result[result["order_item_id"].isnull()]

**🔍 สังเกต:** ผลลัพธ์ควรเจอสินค้า 2 รายการที่ไม่เคยถูกสั่งซื้อเลย (ตามที่ตั้งใจไว้ตอนสร้างข้อมูลจำลอง) — เทคนิคเดียวกับการหาลูกค้าที่ไม่เคยสั่งซื้อ แค่สลับมุมมองมาที่สินค้าแทน

### 4.1 สรุป: เมื่อไหร่ใช้ JOIN แบบไหน

| ต้องการอะไร | ใช้ JOIN แบบไหน |
|---|---|
| เฉพาะแถวที่มีข้อมูลครบทั้งสองตาราง | `INNER JOIN` |
| ทุกแถวจากตารางหลัก แม้บางแถวไม่มีคู่ในอีกตาราง | `LEFT JOIN` |
| ทุกแถวจากตารางที่สอง (ตารางขวา) | `RIGHT JOIN` (หรือสลับตารางใช้ `LEFT JOIN` แทนใน SQLite รุ่นเก่า) |
| หาแถวที่ "ขาด" ความสัมพันธ์ (เช่น ไม่เคยสั่งซื้อ) | `LEFT JOIN` + `WHERE ... IS NULL` |

> 💡 **เกร็ดเสริม:** มี `FULL OUTER JOIN` ด้วย (เก็บทุกแถวจากทั้งสองตาราง) แต่ SQLite ไม่รองรับโดยตรงเช่นกัน ต้องใช้ `LEFT JOIN` รวมกับ `UNION` ของ `RIGHT JOIN` แทน — ไม่ใช้บ่อยเท่า 3 แบบข้างบน จึงไม่ลงรายละเอียดในคาบนี้


---
## 5. Subquery เบื้องต้น

**Subquery (คำสั่งย่อย)** คือ query ที่ซ้อนอยู่ภายใน query อีกตัว — ใช้เมื่อต้องการผลลัพธ์จาก query หนึ่งไปเป็น "เงื่อนไข" หรือ "ข้อมูลนำเข้า" ของอีก query

เทียบกับที่เรียนมา: คล้ายกับการเขียนฟังก์ชันซ้อนฟังก์ชัน (`len(set(my_list))`) — ผลลัพธ์ของ query ด้านในจะถูกใช้โดย query ด้านนอก

### 5.1 Subquery ใน `WHERE` (ใช้บ่อยที่สุด)


In [ ]:
# หาคำสั่งซื้อของลูกค้าที่อยู่ใน Bangkok (ใช้ subquery หา customer_id ก่อน แล้วนำไปกรอง orders)
result = pd.read_sql_query("""
SELECT order_id, customer_id, order_date, status
FROM orders
WHERE customer_id IN (
    SELECT customer_id FROM customers WHERE city = 'Bangkok'
)
ORDER BY order_date
""", conn)
result.head(10)

**อธิบาย:** Subquery ด้านใน (`SELECT customer_id FROM customers WHERE city = 'Bangkok'`) รันก่อน ได้ list ของ `customer_id` ทั้งหมดที่อยู่ Bangkok แล้ว query ด้านนอกใช้ `IN` กรอง `orders` ด้วย list นั้น

**เทียบเท่ากับการเขียนด้วย JOIN:**


In [ ]:
# query เดียวกัน แต่เขียนด้วย JOIN แทน subquery (มักเร็วกว่าในฐานข้อมูลขนาดใหญ่)
result_v2 = pd.read_sql_query("""
SELECT o.order_id, o.customer_id, o.order_date, o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE c.city = 'Bangkok'
ORDER BY o.order_date
""", conn)

print("ผลลัพธ์เหมือนกันไหม:", len(result) == len(result_v2))
result_v2.head(10)

> 💡 **เกร็ดเสริม:** หลายครั้ง Subquery และ JOIN ให้ผลลัพธ์เหมือนกัน — เลือกแบบที่อ่านเข้าใจง่ายกว่าในแต่ละสถานการณ์ โดยทั่วไป **JOIN เร็วกว่าในฐานข้อมูลขนาดใหญ่** เพราะ database engine ปรับแผนการทำงานได้ดีกว่า แต่ **Subquery มักอ่านเข้าใจง่ายกว่า** เมื่อ logic ซับซ้อนหรือต้องเช็คเงื่อนไขที่ซ้อนกันหลายชั้น

### 5.2 Subquery ใน `SELECT` (เรียกว่า Scalar Subquery — คืนค่าเดียว)


In [ ]:
# แสดงราคาสินค้า พร้อมเทียบกับราคาเฉลี่ยของสินค้าทั้งหมด (subquery คืนค่าเดียวมาเทียบ)
result = pd.read_sql_query("""
SELECT product_name, unit_price,
       (SELECT AVG(unit_price) FROM products) AS overall_avg_price,
       unit_price - (SELECT AVG(unit_price) FROM products) AS diff_from_avg
FROM products
ORDER BY diff_from_avg DESC
LIMIT 5
""", conn)
result

### 5.3 Subquery กับ Aggregate Function — หาค่าที่ "สูงกว่าเกณฑ์เฉลี่ย"

รูปแบบที่ใช้บ่อยมาก: หาแถวที่มีค่ามากกว่า/น้อยกว่าค่าเฉลี่ยของทั้งหมด


In [ ]:
# หาสินค้าที่ราคาสูงกว่าราคาเฉลี่ยของสินค้าทั้งหมด
result = pd.read_sql_query("""
SELECT product_name, category, unit_price
FROM products
WHERE unit_price > (SELECT AVG(unit_price) FROM products)
ORDER BY unit_price DESC
""", conn)
print(f"พบสินค้า {len(result)} ชนิด ที่ราคาสูงกว่าค่าเฉลี่ย")
result

### 5.4 สรุป: Subquery vs JOIN vs Subquery ใน HAVING

| ใช้ Subquery เมื่อ... | ใช้ JOIN เมื่อ... |
|---|---|
| ต้องการ "ค่าเดียว" มาเทียบ (เช่น ค่าเฉลี่ยทั้งหมด) | ต้องการ "ดึงคอลัมน์" จากอีกตารางมาแสดง |
| เงื่อนไขซ้อนกันหลายชั้น อ่านเป็นขั้นตอนชัดเจนกว่า | ต้องการประสิทธิภาพสูงสุดกับข้อมูลขนาดใหญ่ |
| ไม่ต้องการคอลัมน์จากตารางอื่น แค่ใช้เป็นเงื่อนไข | ต้องการรวมข้อมูลจริงๆจากหลายตาราง |


---
# ส่วนที่ 2: เชื่อม Python กับ SQLite

## 6. `sqlite3` พื้นฐาน 

### 6.1 วงจรการทำงานกับฐานข้อมูล

```python
conn = sqlite3.connect("ชื่อไฟล์.db")        # 1. เชื่อมต่อ
cursor = conn.cursor()                    # 2. สร้าง cursor (ตัวส่งคำสั่ง SQL)
cursor.execute("...")                     # 3. รันคำสั่ง SQL
conn.commit()                             # 4. บันทึกการเปลี่ยนแปลง (ต้องทำหลัง INSERT/UPDATE/DELETE)
result = cursor.fetchall()                # 5. ดึงผลลัพธ์ (สำหรับ SELECT)
conn.close()                              # 6. ปิดการเชื่อมต่อ เมื่อใช้งานเสร็จ
```

### 6.2 `fetchall()` vs `fetchone()` vs `fetchmany()`

| เมธอด | คืนค่า |
|---|---|
| `.fetchall()` | ทุกแถวที่เหลือ เป็น list ของ tuple |
| `.fetchone()` | แถวเดียว (แถวแรกที่ยังไม่ถูกดึง) เป็น tuple เดียว หรือ `None` ถ้าไม่มีเหลือ |
| `.fetchmany(n)` | n แถว เป็น list ของ tuple |


In [ ]:
cursor.execute("SELECT product_name, unit_price FROM products ORDER BY unit_price DESC")

print("fetchone() ครั้งที่ 1:", cursor.fetchone())   # ได้แถวแรก (แพงที่สุด)
print("fetchone() ครั้งที่ 2:", cursor.fetchone())   # ได้แถวที่ 2 (cursor จำตำแหน่งที่ดึงไปแล้ว)
print("fetchmany(3):", cursor.fetchmany(3))           # ได้อีก 3 แถวถัดไป

### 6.3 `with` statement — จัดการ connection ให้ปิดอัตโนมัติ

วิธีที่ปลอดภัยกว่าการเรียก `.close()` เอง — ถ้าเกิด error กลางทาง connection จะถูกปิดให้อัตโนมัติเสมอ (เทียบกับการเปิดไฟล์ด้วย `with open(...) as f:` ที่อาจเคยเห็นมาก่อน)


In [ ]:
with sqlite3.connect("online_store.db") as temp_conn:
    temp_cursor = temp_conn.cursor()
    temp_cursor.execute("SELECT COUNT(*) FROM customers")
    print("จำนวนลูกค้า:", temp_cursor.fetchone()[0])
# connection ปิดอัตโนมัติเมื่อออกจาก with block (ไม่ต้องเรียก .close() เอง)
print("ออกจาก with block แล้ว - temp_conn ถูกปิดอัตโนมัติ")

---
## 7. `pandas.read_sql_query()` — ได้ DataFrame ตรงจาก SQL

ใช้บ่อยที่สุดในงาน Data Science เพราะได้ผลลัพธ์เป็น **DataFrame ทันที** ไม่ต้องวน loop จัดการ tuple จาก `fetchall()` เอง — ใช้ฟังก์ชัน/เมธอดของ pandas ที่เรียนมาทั้งหมดต่อได้เลย

### โครงสร้าง

```python
df = pd.read_sql_query(sql_query_string, connection)
```


In [ ]:
df = pd.read_sql_query("SELECT * FROM orders", conn)
print(type(df))
df.head()

### 7.1 ใช้ฟังก์ชัน pandas ที่เรียนมาทั้งหมดต่อได้ทันที


In [ ]:
df_orders = pd.read_sql_query("SELECT * FROM orders", conn)

print(df_orders.info())
print()
print(df_orders["status"].value_counts())   # ใช้ value_counts() ที่เรียนมาจาก Session 9 ได้ตรงๆ

### 7.2 รวม SQL query ที่ซับซ้อนเข้ากับ `read_sql_query()` แล้ว plot ต่อทันที (เชื่อมกับ Session 11)


In [ ]:
revenue_by_category = pd.read_sql_query("""
SELECT p.category, SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY total_revenue DESC
""", conn)

plt.figure(figsize=(10, 5))
plt.bar(revenue_by_category["category"], revenue_by_category["total_revenue"], color="steelblue")
plt.title("รายได้รวมแยกตามหมวดหมู่สินค้า (เฉพาะคำสั่งซื้อที่สำเร็จ)")
plt.ylabel("รายได้ (บาท)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**🔍 สังเกต:** นี่คือ workflow ที่ใช้บ่อยมากในงานจริง — เขียน SQL query ที่ซับซ้อน (join 3 ตาราง + filter + group by) ให้ฐานข้อมูลคำนวณให้เสร็จเลย แล้วได้ DataFrame กลับมาพร้อม plot ต่อทันที **ไม่ต้องโหลดข้อมูลดิบทั้งหมดเข้า pandas ก่อนแล้วมาคำนวณเอง** — มีประสิทธิภาพกว่ามากเมื่อฐานข้อมูลมีขนาดใหญ่

### 7.3 `df.to_sql()` — เขียนข้อมูลจาก DataFrame กลับเข้าฐานข้อมูล (ทิศทางตรงข้าม)

บางครั้งหลังประมวลผลข้อมูลด้วย pandas แล้ว อยากบันทึกผลลัพธ์กลับเข้าฐานข้อมูลเป็นตารางใหม่


In [ ]:
revenue_by_category.to_sql("category_revenue_summary", conn, if_exists="replace", index=False)

# ตรวจสอบว่าตารางใหม่ถูกสร้างจริง
check = pd.read_sql_query("SELECT * FROM category_revenue_summary", conn)
print("ตารางใหม่ถูกสร้างจาก DataFrame สำเร็จ:")
check

**พารามิเตอร์ `if_exists`:** `"replace"` (ลบตารางเดิมแล้วสร้างใหม่), `"append"` (เพิ่มแถวเข้าตารางเดิม), `"fail"` (error ถ้าตารางมีอยู่แล้ว — เป็นค่า default)


---
## 8. Parameterized Query — ป้องกัน SQL Injection

ใน Notebook ที่แล้ว เคยเห็นตัวอย่างที่ใช้ f-string ต่อค่าเข้า SQL ตรงๆ (เพื่อสาธิตเท่านั้น) — หัวข้อนี้จะอธิบายว่า**ทำไมวิธีนั้นอันตราย** และวิธีที่ถูกต้อง

### 8.1 ปัญหา: SQL Injection คืออะไร

ลองดูฟังก์ชันค้นหาลูกค้าตามเมืองที่ผู้ใช้ป้อนเอง (จำลองสถานการณ์เว็บแอปที่รับ input จากผู้ใช้)


In [ ]:
def search_customers_unsafe(city_input):
    # ❌ วิธีอันตราย - ต่อค่าที่ผู้ใช้ป้อนเข้า SQL ตรงๆ ด้วย f-string
    query = f"SELECT * FROM customers WHERE city = '{city_input}'"
    print("Query ที่จะรัน:", query)
    return pd.read_sql_query(query, conn)

# การใช้งานปกติ - ทำงานถูกต้อง
result = search_customers_unsafe("Bangkok")
print(f"พบลูกค้า {len(result)} คน\n")

In [ ]:
# ตัวอย่างการโจมตีแบบ SQL Injection — ผู้ใช้ป้อนค่าที่ไม่ใช่ชื่อเมืองปกติ แต่เป็นโค้ด SQL
malicious_input = "Bangkok' OR '1'='1"
result = search_customers_unsafe(malicious_input)
print(f"พบลูกค้า {len(result)} คน (ควรพบแค่คนที่อยู่ Bangkok แต่กลับได้ทุกคน!)")

**เกิดอะไรขึ้น:** ค่าที่ผู้ใช้ป้อน `Bangkok' OR '1'='1` ทำให้ query กลายเป็น:
```sql
SELECT * FROM customers WHERE city = 'Bangkok' OR '1'='1'
```
เงื่อนไข `'1'='1'` เป็นจริงเสมอ จึงทำให้ query คืนค่า**ทุกแถว**ในตาราง ไม่ใช่แค่ลูกค้าที่อยู่ Bangkok ตามที่ตั้งใจ — นี่คือตัวอย่างเบื้องต้นของ **SQL Injection** ในการโจมตีจริง ผู้ไม่หวังดีอาจป้อนคำสั่งที่ร้ายแรงกว่านี้มาก (เช่น ลบทั้งตาราง หรือดึงข้อมูลที่ไม่ควรเข้าถึงได้)

### 8.2 วิธีแก้: Parameterized Query (ใช้ `?` แทนการต่อ string)

เหมือนที่เคยใช้กับ `executemany()` ใน Notebook ที่แล้ว — ใช้ `?` เป็นตัวแทนค่า แล้วส่งค่าจริงแยกเป็น tuple ต่างหาก ฐานข้อมูลจะ**แยกแยะค่าข้อมูลกับคำสั่ง SQL ออกจากกันอย่างเด็ดขาด** ทำให้ค่าที่ป้อนเข้ามาไม่ว่าจะเป็นอะไรก็ไม่ถูกตีความเป็นโค้ด SQL


In [ ]:
def search_customers_safe(city_input):
    # ✅ วิธีที่ถูกต้อง - ใช้ ? แทนค่า แล้วส่งค่าจริงแยกเป็น tuple พารามิเตอร์ที่ 2
    query = "SELECT * FROM customers WHERE city = ?"
    return pd.read_sql_query(query, conn, params=(city_input,))

# ลองโจมตีแบบเดียวกันอีกครั้ง - ครั้งนี้ปลอดภัย
malicious_input = "Bangkok' OR '1'='1"
result = search_customers_safe(malicious_input)
print(f"พบลูกค้า {len(result)} คน (ปลอดภัยแล้ว - ไม่มีลูกค้าที่ city เท่ากับ string แปลกๆนี้จริง)")

# ลองใช้งานปกติ - ยังทำงานถูกต้องเหมือนเดิม
result_normal = search_customers_safe("Bangkok")
print(f"ค้นหา 'Bangkok' ปกติ: พบ {len(result_normal)} คน")

### 8.3 Parameterized Query กับ `sqlite3` โดยตรง (ไม่ผ่าน pandas)

ใช้ `?` แบบเดียวกันกับ `cursor.execute()` ได้เลย


In [ ]:
cursor.execute("SELECT * FROM products WHERE category = ? AND unit_price > ?", ("เสื้อผ้า", 300))
results = cursor.fetchall()
for row in results:
    print(row)

### 8.4 Parameterized Query กับหลายค่า (Named Placeholders)

ถ้ามีพารามิเตอร์หลายตัว ใช้ `:ชื่อ` แทน `?` ช่วยให้อ่าน query เข้าใจง่ายขึ้นเมื่อมีพารามิเตอร์จำนวนมาก


In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
WHERE category = :cat AND unit_price BETWEEN :min_price AND :max_price
"""
result = pd.read_sql_query(query, conn, params={"cat": "อิเล็กทรอนิกส์", "min_price": 200, "max_price": 1000})
result

### 8.5 สรุปกฎเหล็ก

> ⚠️ **กฎเหล็กของการเขียน SQL ร่วมกับค่าจาก "ผู้ใช้" หรือ "ตัวแปรภายนอก": ห้ามต่อค่าเข้า SQL string ตรงๆ ด้วย f-string/`.format()`/`+` เด็ดขาด ให้ใช้ `?` หรือ `:name` แบบ parameterized query เสมอ** ไม่ว่าจะดูเหมือนปลอดภัยแค่ไหนก็ตาม เพราะค่าที่มาจากภายนอก (ผู้ใช้กรอกฟอร์ม, ไฟล์ที่อัปโหลด, API ภายนอก) ไม่สามารถเชื่อถือได้ว่าจะไม่มีโค้ดอันตรายปนมา


---
# ส่วนที่ 3: End-to-end Mini-Exercise

## 9. DB → SQL → DataFrame → Clean → Visualize 

ถึงเวลารวมทุกอย่างที่เรียนมาตลอดทั้งคอร์สเข้าด้วยกัน — Notebook นี้จะเดินผ่าน **workflow ทำงานกับข้อมูลแบบครบวงจร** ที่ใช้จริงในงาน Data Analyst/Data Scientist:

```
ฐานข้อมูล (SQLite) → SQL Query (JOIN + filter) → DataFrame (pandas) → ทำความสะอาด → สร้างกราฟ (insight)
```

**คำถามทางธุรกิจที่จะตอบ:** *"ลูกค้ากลุ่มไหนสร้างรายได้ให้ร้านมากที่สุด และมีอะไรที่น่าสนใจเกี่ยวกับพฤติกรรมการซื้อของพวกเขา?"*

### ขั้นที่ 1: ดึงข้อมูลจากฐานข้อมูลด้วย SQL (JOIN หลายตาราง)


In [ ]:
# JOIN 4 ตารางเข้าด้วยกัน เพื่อให้ได้ข้อมูลระดับ "รายการสินค้าในคำสั่งซื้อ" พร้อมข้อมูลลูกค้าและสินค้าครบ
raw_data = pd.read_sql_query("""
SELECT
    c.customer_id, c.first_name, c.last_name, c.city, c.signup_date,
    o.order_id, o.order_date, o.status,
    p.product_name, p.category,
    oi.quantity, oi.unit_price
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON oi.product_id = p.product_id
""", conn)

print("ขนาดข้อมูลที่ได้:", raw_data.shape)
raw_data.head()

### ขั้นที่ 2: ทำความสะอาดข้อมูล 


In [ ]:
# ตรวจสอบ missing values ก่อน
print("Missing values:")
print(raw_data.isnull().sum())

# สร้างคอลัมน์ยอดเงินรวมต่อรายการ (quantity * unit_price)
raw_data["line_total"] = raw_data["quantity"] * raw_data["unit_price"]

# แปลงวันที่เป็น datetime (เทคนิคจาก Session 8)
raw_data["order_date"] = pd.to_datetime(raw_data["order_date"])
raw_data["signup_date"] = pd.to_datetime(raw_data["signup_date"])

# กรองเฉพาะคำสั่งซื้อที่สำเร็จ (completed) เท่านั้น สำหรับวิเคราะห์รายได้จริง
# (คำสั่งซื้อที่ cancelled/pending ยังไม่ควรนับเป็นรายได้จริง)
clean_data = raw_data[raw_data["status"] == "completed"].copy()

print(f"\nข้อมูลทั้งหมด: {len(raw_data)} แถว")
print(f"เฉพาะคำสั่งซื้อที่สำเร็จ: {len(clean_data)} แถว")

### ขั้นที่ 3: สรุปข้อมูลด้วย `groupby()` + `agg()` 


In [ ]:
customer_summary = clean_data.groupby(["customer_id", "first_name", "last_name", "city"]).agg(
    total_revenue=("line_total", "sum"),
    num_orders=("order_id", "nunique"),   # nunique = จำนวนค่าไม่ซ้ำ (จำนวนคำสั่งซื้อที่แตกต่างกัน)
    num_items=("quantity", "sum"),
).reset_index()

customer_summary = customer_summary.sort_values("total_revenue", ascending=False)
print("ลูกค้าที่สร้างรายได้สูงสุด 10 อันดับแรก:")
customer_summary.head(10)

### ขั้นที่ 4: สร้าง Feature เพิ่มเติม 

In [ ]:
# คำนวณยอดซื้อเฉลี่ยต่อคำสั่งซื้อ (Average Order Value)
customer_summary["avg_order_value"] = customer_summary["total_revenue"] / customer_summary["num_orders"]

# แบ่งกลุ่มลูกค้าตามรายได้ (binning - เทคนิคจาก Data Cleaning)
# ใช้เกณฑ์ที่อ้างอิงจากการกระจายตัวจริงของข้อมูล (ลองดู .describe() ก่อนตั้งเกณฑ์เสมอ)
def categorize_customer(revenue):
    if revenue >= 14000:
        return "VIP"
    elif revenue >= 4000:
        return "ลูกค้าประจำ"
    else:
        return "ลูกค้าทั่วไป"

customer_summary["customer_tier"] = customer_summary["total_revenue"].apply(categorize_customer)

print(customer_summary["customer_tier"].value_counts())
customer_summary.head(10)

### ขั้นที่ 5: Visualize เพื่อหา Insight 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# กราฟ 1: Top 10 ลูกค้าที่สร้างรายได้สูงสุด
top10 = customer_summary.head(10)
colors = ["firebrick" if tier == "VIP" else "steelblue" for tier in top10["customer_tier"]]
axes[0].barh(top10["first_name"] + " " + top10["last_name"], top10["total_revenue"], color=colors)
axes[0].invert_yaxis()
axes[0].set_title("Top 10 ลูกค้าที่สร้างรายได้สูงสุด")
axes[0].set_xlabel("รายได้รวม (บาท)")

# กราฟ 2: สัดส่วนลูกค้าตาม tier
tier_counts = customer_summary["customer_tier"].value_counts()
axes[1].bar(tier_counts.index, tier_counts.values, color=["gold", "silver", "lightgray"])
axes[1].set_title("จำนวนลูกค้าแยกตามระดับ (Tier)")
axes[1].set_ylabel("จำนวนลูกค้า")

plt.tight_layout()
plt.show()

In [ ]:
# กราฟ 3: ความสัมพันธ์ระหว่างจำนวนคำสั่งซื้อ กับ ยอดซื้อเฉลี่ยต่อครั้ง
plt.figure(figsize=(8, 6))
tier_colors = {"VIP": "firebrick", "ลูกค้าประจำ": "steelblue", "ลูกค้าทั่วไป": "lightgray"}
for tier, color in tier_colors.items():
    subset = customer_summary[customer_summary["customer_tier"] == tier]
    plt.scatter(subset["num_orders"], subset["avg_order_value"], label=tier, color=color, alpha=0.7, s=80)

plt.title("จำนวนคำสั่งซื้อ vs ยอดซื้อเฉลี่ยต่อครั้ง แยกตามระดับลูกค้า")
plt.xlabel("จำนวนคำสั่งซื้อ")
plt.ylabel("ยอดซื้อเฉลี่ยต่อครั้ง (บาท)")
plt.legend(title="ระดับลูกค้า")
plt.tight_layout()
plt.show()

### ขั้นที่ 6: สรุปผล (Storytelling)


In [ ]:
vip_count = (customer_summary["customer_tier"] == "VIP").sum()
vip_revenue_share = customer_summary[customer_summary["customer_tier"] == "VIP"]["total_revenue"].sum() / customer_summary["total_revenue"].sum() * 100

top_customer = customer_summary.iloc[0]

print("สรุปผลการวิเคราะห์:")
print(f"1. มีลูกค้า VIP (รายได้ >= 14,000 บาท) จำนวน {vip_count} คน จากทั้งหมด {len(customer_summary)} คน")
print(f"2. ลูกค้า VIP สร้างรายได้รวม {vip_revenue_share:.1f}% ของรายได้ทั้งหมด")
print(f"3. ลูกค้าที่สร้างรายได้สูงสุดคือ {top_customer['first_name']} {top_customer['last_name']} "
      f"({top_customer['total_revenue']:.0f} บาท จาก {top_customer['num_orders']} คำสั่งซื้อ)")
print(f"\nข้อเสนอแนะ: ควรพิจารณาทำโปรแกรมสมาชิก/สิทธิประโยชน์พิเศษให้กลุ่มลูกค้า VIP "
      f"เพื่อรักษาฐานลูกค้ากลุ่มนี้ไว้ เนื่องจากสร้างรายได้ในสัดส่วนที่สูงมากเมื่อเทียบกับจำนวนคน")

**🎯 สรุป Mini-Exercise นี้:** ใช้ทักษะจากแทบทุกคาบที่เรียนมาในไฟล์เดียว — SQL JOIN (4 ตาราง), pandas cleaning, `groupby()`+`agg()`, `apply()` สร้าง feature ใหม่, และ matplotlib visualization พร้อม storytelling สรุปผล — นี่คือรูปแบบงานที่ Data Analyst/Data Scientist ทำเป็นประจำในการทำงานจริง


---
## 🧪 แบบฝึกหัดท้ายเรื่อง

> ลองทำเองในเซลล์ที่เตรียมไว้ด้านล่างแต่ละข้อ — ถ้าไม่แน่ใจให้ลองรันดูผลลัพธ์ หรือถามอาจารย์/เพื่อนได้เลย ใช้ฐานข้อมูล `online_store.db` ตลอดทั้งชุดแบบฝึกหัดนี้

### ข้อ 1: INNER JOIN
เขียน query (ใช้ `pd.read_sql_query()`) หา:
1. รายการสินค้าทั้งหมดที่อยู่ในคำสั่งซื้อที่มีสถานะ `"completed"` (JOIN `order_items`, `orders`, `products`)
2. ชื่อลูกค้าและชื่อสินค้าที่ลูกค้าคนนั้นซื้อ (JOIN 4 ตารางเหมือนใน mini-exercise แต่ลองเขียนเองดู)


In [ ]:
# เขียนคำตอบข้อ 1 ที่นี่


### ข้อ 2: LEFT JOIN
เขียน query หา:
1. สินค้าทั้งหมดที่ไม่มีอยู่ในตาราง `category_revenue_summary` (ตารางที่สร้างจาก `to_sql()` ตอนต้นไฟล์) — ใช้ `LEFT JOIN` แล้วกรอง `IS NULL`
2. ลูกค้าทุกคนพร้อมจำนวนคำสั่งซื้อของแต่ละคน (รวมลูกค้าที่มี 0 คำสั่งซื้อด้วย — ใช้ `LEFT JOIN` + `GROUP BY` + `COUNT()`)


In [ ]:
# เขียนคำตอบข้อ 2 ที่นี่


### ข้อ 3: Subquery
เขียน query หา:
1. ลูกค้าที่มี `total spending` (ผลรวม quantity*unit_price จากคำสั่งซื้อที่ completed) มากกว่าค่าเฉลี่ยของลูกค้าทั้งหมด
2. สินค้าที่มีราคาต่ำกว่าราคาเฉลี่ยของหมวดหมู่ตัวเอง (Hint: subquery ต้องกรองตาม category ด้วย ลองใช้ correlated subquery หรือคิดวิธีอื่นที่ทำได้)


In [ ]:
# เขียนคำตอบข้อ 3 ที่นี่


### ข้อ 4: Parameterized Query
เขียนฟังก์ชัน Python `get_orders_by_status(status_input)` ที่ใช้ parameterized query ดึงคำสั่งซื้อทั้งหมดที่มีสถานะตรงกับ `status_input` แล้วทดสอบเรียกใช้ด้วยค่า `"completed"`, `"pending"`, และค่าที่พยายามทำ SQL Injection (เช่น `"completed' OR '1'='1"`) เพื่อยืนยันว่าฟังก์ชันปลอดภัย


In [ ]:
# เขียนคำตอบข้อ 4 ที่นี่


### ข้อ 5: ท้าทายเพิ่มเติม (Challenge) — Mini end-to-end ของตัวเอง
เลือกคำถามทางธุรกิจของตัวเอง 1 คำถามเกี่ยวกับฐานข้อมูลนี้ (ตัวอย่าง: "หมวดหมู่สินค้าไหนขายดีที่สุดในแต่ละเดือน", "เมืองไหนมีลูกค้าที่ใช้จ่ายเฉลี่ยสูงสุด") แล้วทำตาม workflow แบบเดียวกับ mini-exercise:
1. เขียน SQL query (ใช้ JOIN/GROUP BY/subquery ตามความเหมาะสม) ดึงข้อมูลที่ต้องการ
2. โหลดเป็น DataFrame ด้วย `pd.read_sql_query()`
3. ทำความสะอาด/สร้าง feature เพิ่มเติมถ้าจำเป็น
4. สร้างกราฟอย่างน้อย 1 อันที่ตอบคำถามนั้น (ตั้งชื่อกราฟให้สื่อความหมาย ตามที่เรียนมาจาก Session 11)
5. เขียนสรุปผล 2-3 ประโยค


In [ ]:
# เขียนคำตอบข้อ 5 ที่นี่


In [ ]:
# ปิดการเชื่อมต่อฐานข้อมูลก่อนจบไฟล์
conn.close()
print("ปิดการเชื่อมต่อฐานข้อมูลแล้ว")

---
## 🔗 เชื่อม Colab กับ GitHub

เก็บ Notebook นี้ (พร้อมไฟล์ `online_store.db`) ขึ้น GitHub ต่อจากไฟล์ก่อนหน้า

### วิธีที่ 1: เซฟจาก Colab ขึ้น GitHub ตรงๆ (สำหรับ Notebook)

1. ใน Colab ไปที่เมนู **File → Save a copy in GitHub**
2. เลือก repository เดิมที่ใช้เก็บ Notebook คาบก่อนๆ (เช่น `SC612104-coursework`)
3. ตั้งชื่อไฟล์และ commit message (เช่น `"เพิ่ม notebook: SQL JOIN, subquery, Python+SQLite"`) แล้วกด **OK**

**⚠️ ข้อควรรู้:** วิธีนี้ push แค่ตัว `.ipynb` — ไฟล์ `online_store.db` ต้องอัปโหลดขึ้น GitHub แยกต่างหาก

### วิธีที่ 2: ใช้ Git ผ่าน Terminal (push ได้ทั้ง Notebook และไฟล์ฐานข้อมูล)

```bash
git clone https://github.com/<username>/<repo-name>.git
cd <repo-name>
git add notebook.ipynb online_store.db
git commit -m "เพิ่ม notebook และฐานข้อมูล: SQL advanced (JOIN, subquery, Python+SQLite)"
git push origin main
```

> 💡 **Tip:** ทักษะ SQL ที่เรียนมา(CREATE TABLE → SELECT → JOIN → Python integration) คือพื้นฐานสำคัญที่สุดอันหนึ่งในงาน Data Analyst/Data Scientist เกือบทุกตำแหน่ง เพราะข้อมูลขององค์กรส่วนใหญ่อยู่ในฐานข้อมูล ไม่ใช่ไฟล์ CSV ที่พร้อมใช้ทันที — ความสามารถในการดึงและรวมข้อมูลจากฐานข้อมูลด้วย SQL ก่อนนำมาวิเคราะห์ด้วย pandas คือทักษะที่ใช้งานจริงทุกวัน
